In [1]:
import math
import random
from dataclasses import dataclass
from typing import Callable, Literal

import einops
import torch
import torch.nn as nn
import torch.nn.functional as F


In [2]:
def set_seed(seed: int = 0):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def get_device():
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


# Data

In [3]:
def gen_prompts(batch_size: int, seq_len: int, vocab_size: int, device) -> torch.Tensor:
    return torch.randint(low=0, high=vocab_size, size=(batch_size, seq_len), device=device)


def sorted_ground_truth(prompt: torch.Tensor) -> torch.Tensor:
    return torch.sort(prompt, dim=-1).values


# Reward Definition

In [4]:
def sort_distance_reward(prompt: list[int], response: list[int]) -> float:
    assert len(prompt) == len(response)
    ground_truth = sorted(prompt)
    return sum(1 for x, y in zip(response, ground_truth) if x == y)


In [5]:
def sort_inclusion_ordering_reward(prompt: list[int], response: list[int]) -> float:
    """
    Return how close response is to ground_truth = sorted(prompt).
    """
    assert len(prompt) == len(response)
    # Give one point for each token in the prompt that shows up in the response
    inclusion_reward = sum(1 for x in prompt if x in response)  # @inspect inclusion_reward
    # Give one point for each adjacent pair in response that's sorted
    ordering_reward = sum(1 for x, y in zip(response, response[1:]) if x <= y)  # @inspect ordering_reward
    return inclusion_reward + ordering_reward


In [6]:
def compute_reward(
    prompts: torch.Tensor, responses: torch.Tensor, reward_fn: Callable[[list[int], list[int]], float]
) -> torch.Tensor:
    """
    Args:
        prompts (int[batch pos])
        responses (int[batch trial pos])
    Returns:
        rewards (float[batch trial])
    """
    batch_size, num_responses, _ = responses.shape
    rewards = torch.empty(batch_size, num_responses, dtype=torch.float32)
    for i in range(batch_size):
        for j in range(num_responses):
            rewards[i, j] = reward_fn(prompts[i, :], responses[i, j, :])
    return rewards


In [7]:
reward_1 = sort_distance_reward([3, 1, 0, 2], [0, 1, 2, 3])
reward_2 = sort_inclusion_ordering_reward([3, 1, 0, 2], [0, 2, 1, 3])
print(f"Distance reward: {reward_1}, Inclusion + Ordering reward: {reward_2}")


Distance reward: 4, Inclusion + Ordering reward: 6


# Model 

In [8]:
class ToySortPolicy(nn.Module):
    def __init__(self, vocab_size: int, embedding_dim: int, prompt_length: int, response_length: int):
        super().__init__()

        self.embedding_dim = embedding_dim
        self.emb = nn.Embedding(vocab_size, embedding_dim)
        self.enc = nn.Parameter(
            torch.randn(prompt_length, embedding_dim, embedding_dim) / math.sqrt(embedding_dim)
        )
        self.dec = nn.Parameter(
            torch.randn(response_length, embedding_dim, embedding_dim) / math.sqrt(embedding_dim)
        )
        self.out = nn.Linear(embedding_dim, vocab_size)

    def forward(self, prompt):
        x = self.emb(prompt)  # (B,L,d1)
        h = torch.einsum("bld,ldm->bm", x, self.enc)  # (B,d2)
        z = torch.einsum("bm,lmd->bld", h, self.dec)  # (B,L,d1)
        return self.out(z)  # (B,L,V)


# Algorithm

In [9]:
@torch.no_grad()
def sample_responses(
    model: ToySortPolicy,
    prompts: torch.Tensor,
    num_responses: int = 4,
    temperature: float = 1.0,
) -> torch.Tensor:
    logits = model(prompts)  # (B,L,V)
    logits = model(prompts)  # [batch pos vocab]
    batch_size = prompts.shape[0]
    # Sample num_responses (independently) for each [batch pos]
    flattened_logits = einops.rearrange(logits, "batch pos vocab -> (batch pos) vocab")
    flattened_responses = torch.multinomial(
        F.softmax(flattened_logits, dim=-1), num_samples=num_responses, replacement=True
    )  # [batch pos trial]
    responses = einops.rearrange(
        flattened_responses, "(batch pos) trial -> batch trial pos", batch=batch_size
    )
    return responses


def compute_log_probs(prompts: torch.Tensor, responses: torch.Tensor, model: ToySortPolicy) -> torch.Tensor:
    logits = model(prompts)
    log_probs = F.log_softmax(logits, dim=-1)  # (B,L,V)

    # Replicate log_probs for each response
    num_responses = responses.shape[1]
    log_probs = einops.repeat(log_probs, "B L V -> B R L V", R=num_responses)

    # Gather log_probs of the sampled responses
    logp_token = torch.gather(log_probs, dim=-1, index=responses.unsqueeze(-1)).squeeze(-1)  # (B,R,L)
    return logp_token


def compute_deltas(
    rewards, mode: Literal["raw", "centered_rewards", "normalized_rewards", "max_rewards"], eps=1e-6
):
    if mode == "raw":
        return rewards
    if mode == "centered_rewards":
        return rewards - rewards.mean(dim=-1, keepdim=True)
    if mode == "normalized_rewards":
        mean_rewards = rewards.mean(dim=-1, keepdim=True)
        std_rewards = rewards.std(dim=-1, keepdim=True)
        centered_rewards = rewards - mean_rewards
        normalized_rewards = centered_rewards / (std_rewards + eps)
        return normalized_rewards
    if mode == "max_rewards":
        # Zero out any reward that isn't the maximum for each batch
        max_rewards = rewards.max(dim=-1, keepdim=True)[0]
        max_rewards = torch.where(rewards == max_rewards, rewards, torch.zeros_like(rewards))
        return max_rewards

    raise ValueError(f"Unknown delta computation mode: {mode}")


def compute_loss(
    log_probs: torch.Tensor,
    old_log_probs: torch.Tensor,
    deltas: torch.Tensor,
    mode: Literal["naive", "unclipped", "clipped"],
) -> torch.Tensor:
    if mode == "naive":
        return -einops.einsum(log_probs, deltas, "batch trial pos, batch trial -> batch trial pos").mean()

    if mode == "unclipped":
        ratios = torch.exp(log_probs - old_log_probs)
        return -einops.einsum(ratios, deltas, "batch trial pos, batch trial -> batch trial pos").mean()

    if mode == "clipped":
        epsilon = 0.01
        unclipped_ratios = torch.exp(log_probs - old_log_probs)
        unclipped = einops.einsum(unclipped_ratios, deltas, "batch trial pos, batch trial -> batch trial pos")
        clipped_ratios = torch.clamp(unclipped_ratios, min=1 - epsilon, max=1 + epsilon)
        clipped = einops.einsum(clipped_ratios, deltas, "batch trial pos, batch trial -> batch trial pos")
        return -torch.minimum(unclipped, clipped).mean()
    raise ValueError(f"Unknown mode: {mode}")


def compute_kl_penalty(
    log_probs: torch.Tensor,
    old_log_probs: torch.Tensor,
) -> torch.Tensor:
    return (
        (torch.exp(old_log_probs - log_probs) - (old_log_probs - log_probs) - 1.0).sum(dim=-1).mean()
    )  # (B,R) -> scalar


In [10]:
@dataclass
class GRPOCfg:
    V: int = 10  # vocab size (token values 0..V-1)
    L: int = 4  # sequence length
    B: int = 128  # batch prompts per outer iter
    K: int = 8  # responses per prompt (group size)
    outer_iters: int = 200
    inner_steps: int = 4

    lr: float = 2e-2
    temperature: float = 1.0

    delta_mode: Literal["raw", "centered_rewards", "normalized_rewards", "max_rewards"] = "centered_rewards"
    loss_mode: Literal["naive", "unclipped", "clipped"] = "clipped"
    clip_eps: float = 0.2

    use_kl: bool = True
    kl_beta: float = 0.02
    ref_update_every: int = 30

    seed: int = 0
    print_every: int = 20

    reward_fn: Literal["distance", "inclusion_ordering"] = "inclusion_ordering"


In [11]:
REWARD_FN_MAP = {
    "distance": sort_distance_reward,
    "inclusion_ordering": sort_inclusion_ordering_reward,
}

In [12]:
def grpo(cfg: GRPOCfg = GRPOCfg()):
    dev = get_device()
    set_seed(cfg.seed)

    policy = ToySortPolicy(cfg.V, cfg.L, cfg.L, cfg.L).to(dev)
    opt = torch.optim.Adam(policy.parameters(), lr=cfg.lr)

    reward_fn = REWARD_FN_MAP[cfg.reward_fn]

    # reference model (slow-moving anchor for KL)
    ref = ToySortPolicy(cfg.V, cfg.L, cfg.L, cfg.L).to(dev)
    ref.load_state_dict(policy.state_dict())
    ref.eval()

    for it in range(cfg.outer_iters):
        # update reference model slowly
        if cfg.use_kl and it > 0 and (it % cfg.ref_update_every == 0):
            ref.load_state_dict(policy.state_dict())
            ref.eval()

        # ---------- OUTER: rollout + rewards + deltas + freeze old/ref logps
        prompt = gen_prompts(cfg.B, cfg.L, cfg.V, dev)  # (B,L)

        with torch.no_grad():
            responses = sample_responses(policy, prompt, cfg.K, cfg.temperature)  # (B,K,L)
            rewards = compute_reward(prompt, responses, reward_fn)  # (B,K)
            deltas = compute_deltas(rewards, cfg.delta_mode)  # (B,K)

            # IMPORTANT: freeze old logps (detach)
            old_logp_token = compute_log_probs(prompt, responses, policy).detach()  # (B,K,L)

            # freeze ref logps for KL (detach)
            ref_logp_token = compute_log_probs(prompt, responses, ref).detach() if cfg.use_kl else None

        # ---------- INNER: multiple gradient steps on same samples
        for _ in range(cfg.inner_steps):
            opt.zero_grad(set_to_none=True)

            logp_token = compute_log_probs(prompt, responses, policy)  # (B,K,L)

            loss = compute_loss(
                log_probs=logp_token,
                old_log_probs=old_logp_token,
                deltas=deltas,
                mode=cfg.loss_mode,
            )

            if cfg.use_kl:
                loss = loss + cfg.kl_beta * compute_kl_penalty(logp_token, ref_logp_token)

            loss.backward()
            opt.step()

        if (it % cfg.print_every) == 0 or it == cfg.outer_iters - 1:
            with torch.no_grad():
                print(
                    f"iter {it:04d} | "
                    f"reward mean {rewards.mean().item():.3f} "
                    f"(min {rewards.min().item():.1f}, max {rewards.max().item():.1f}) | "
                    f"loss {loss.item():.4f}"
                )

    return policy


In [13]:
def sample_and_evaluate(policy: ToySortPolicy, cfg):
    policy.eval()
    dev = get_device()

    with torch.no_grad():
        p = gen_prompts(8, cfg.L, cfg.V, dev)
        r = sample_responses(policy, p, num_responses=1, temperature=0.8).squeeze(1)
        gt = sorted_ground_truth(p)
        print("\n=== sample check ===")
        for i in range(p.size(0)):
            print(
                f"prompt: {p[i].tolist()} | pred: {r[i].tolist()} | gt: {gt[i].tolist()} | sort reward: {sort_distance_reward(p[i].tolist(), r[i].tolist())} | inclusion+ordering reward: {sort_inclusion_ordering_reward(p[i].tolist(), r[i].tolist())}"
            )

In [14]:
cfg = GRPOCfg()
policy = grpo(cfg)


iter 0000 | reward mean 3.070 (min 0.0, max 7.0) | loss 0.0138
iter 0020 | reward mean 3.250 (min 0.0, max 6.0) | loss 0.0034
iter 0040 | reward mean 3.396 (min 0.0, max 7.0) | loss 0.0008
iter 0060 | reward mean 3.625 (min 0.0, max 7.0) | loss 0.0016
iter 0080 | reward mean 3.813 (min 1.0, max 7.0) | loss 0.0059
iter 0100 | reward mean 4.071 (min 1.0, max 7.0) | loss 0.0043
iter 0120 | reward mean 4.278 (min 1.0, max 7.0) | loss 0.0030
iter 0140 | reward mean 4.497 (min 1.0, max 7.0) | loss 0.0058
iter 0160 | reward mean 4.710 (min 1.0, max 7.0) | loss 0.0058
iter 0180 | reward mean 4.859 (min 2.0, max 7.0) | loss 0.0033
iter 0199 | reward mean 4.964 (min 2.0, max 7.0) | loss 0.0067


In [16]:
sample_and_evaluate(policy, cfg)  # inclusion+ordering reward



=== sample check ===
prompt: [5, 0, 0, 3] | pred: [0, 2, 4, 6] | gt: [0, 0, 3, 5] | sort reward: 1 | inclusion+ordering reward: 5
prompt: [9, 0, 0, 8] | pred: [6, 9, 9, 8] | gt: [0, 0, 8, 9] | sort reward: 0 | inclusion+ordering reward: 4
prompt: [0, 2, 1, 4] | pred: [0, 4, 7, 9] | gt: [0, 1, 2, 4] | sort reward: 1 | inclusion+ordering reward: 5
prompt: [8, 3, 0, 7] | pred: [1, 3, 7, 8] | gt: [0, 3, 7, 8] | sort reward: 3 | inclusion+ordering reward: 6
prompt: [7, 8, 1, 7] | pred: [1, 7, 1, 8] | gt: [1, 7, 7, 8] | sort reward: 3 | inclusion+ordering reward: 6
prompt: [6, 0, 2, 0] | pred: [0, 5, 6, 6] | gt: [0, 0, 2, 6] | sort reward: 2 | inclusion+ordering reward: 6
prompt: [4, 2, 4, 1] | pred: [0, 1, 4, 7] | gt: [1, 2, 4, 4] | sort reward: 1 | inclusion+ordering reward: 6
prompt: [5, 2, 4, 4] | pred: [0, 5, 4, 4] | gt: [2, 4, 4, 5] | sort reward: 1 | inclusion+ordering reward: 5
